# Handoff and Customer Success: Transfer the Capability, Not the Folder

> In the late 1980s, the UK government's Central Computer and Telecommunications Agency developed the practices that became ITIL to make service operation repeatable across organizations rather than dependent on one implementer. Riverside has the same problem at a smaller scale: a design is not an operable service until authority, evidence, and recovery knowledge survive the FDE's departure.
>
> **Where you are:** The frozen Riverside case has reached a broad-availability gate that requires handoff acceptance and recurring ownership. The platform repository contains substantial source assets and procedures, but no executed cloud evidence, committed SLO, rehearsed rollback, or accepted post-hypercare quality owner. This notebook keeps those gaps visible.
>
> **Notation:** `HOF-*` - handoff artifact; `OPS-*` - operating artifact; `RBK-*` - runbook; `DRL-*` - drill; `CHG-*` - change record; `[Measured]` - observed evidence; `[Modeled]` - assumption-based result; `[Customer-validated]` - bounded decision by an authorized customer role.

| Part | Decision to earn | Output |
|---:|---|---|
| 0 | Why does the polished document dump fail? | Failed package assessment |
| 1 | What makes a package traceable? | Package index |
| 2 | Is the service ready for ownership? | Readiness review |
| 3 | Which signals produce which actions? | Dashboard and alert guide |
| 4 | Can operators contain and recover safely? | Runbook set |
| 5 | How may policy behavior and thresholds change? | Change process |
| 6 | Who responds, when, and with what authority? | Support matrix |
| 7 | Can named operators perform the work? | Training and drills |
| 8 | What blocks acceptance and FDE exit? | Sign-off and exit gates |
| 9 | How does ownership persist after hypercare? | Health review and backlog |
| 10 | What can this module honestly claim? | Limitations and honest close |

## 0 · The Challenge

> **The mission**: Riverside House - transfer monitoring, response, change, support, training, acceptance, and recurring health ownership without leaving the FDE as an undocumented production dependency.

**What we know so far:**

- The frozen case defines weekday support hours, a 30-day hypercare input, severity targets, capability ownership, seeded incidents, and broad-availability gates.
- The platform repository defines intended architecture, bounded telemetry, rollback and incident procedures, and an evidence ledger.
- **But no executed evidence proves that the receiving team can use any of it.**

**What's blocking us:**

The proposed handoff is a polished folder containing a dashboard export, generic runbook, support spreadsheet, training slides, and signature page. During the first alert, the operator cannot identify the accountable owner, safe first action, escalation authority, evidence to preserve, or re-enablement gate. Riverside has documents and still does not have an operating capability.

**What this chapter unlocks:**

A traceable package in which each recurring task and alert maps to an owner, decision, runbook, evidence record, drill, limitation, and revalidation trigger.

**Predict:** Riverside received every file named in the checklist. Which outcome should the first review produce?

1. **Presence pass** - artifact presence proves completeness.
2. **FDE-dependent pass** - accept because the FDE can remain available for questions.
3. **Capability failure** - reject because no capability is linked to an owner, action, evidence, drill, limitation, or revalidation trigger.

Choose one named outcome before reading the validator. The next cell must report whether your choice matches the required capability fields.

In [ ]:
# ── Fail the Document-Dump Handoff ───────────────────────────────────────────
REQUIRED_CAPABILITY_FIELDS = {
    "artifact_id", "capability", "receiving_owner", "decision_or_action",
    "evidence_ref", "drill_status", "limitations", "revalidate_on",
}

def assess_handoff(records):
    issues = []
    for position, record in enumerate(records, start=1):
        missing = sorted(REQUIRED_CAPABILITY_FIELDS - set(record))
        if missing:
            issues.append({"record": position, "missing": missing})
    return issues

document_dump = [
    {"file": "dashboard.pdf"},
    {"file": "runbook.docx"},
    {"file": "support.xlsx"},
    {"file": "training-slides.pptx"},
    {"file": "acceptance.pdf"},
]
document_dump_issues = assess_handoff(document_dump)
actual_outcome = "Capability failure" if document_dump_issues else "Presence pass"
print(f"FAIL: {len(document_dump_issues)} of {len(document_dump)} records lack capability-transfer fields.")
for issue in document_dump_issues:
    print(f"  record {issue['record']}: missing {', '.join(issue['missing'])}")
print(f"Prediction closed: {actual_outcome} - filenames cannot prove ownership or safe action.")
print("  Takeaway: file presence is not operating evidence.")

#### What failed, and why it matters

The validator is intentionally unimpressed by filenames. A receiving operator needs a path from symptom to decision, authority, action, evidence, and recovery.

```mermaid
flowchart LR
    D["Document exists"] --> C{"Capability mapped?"}
    C -- No --> F["Handoff remains open"]
    C -- Yes --> O["Named owner"]
    O --> A["Decision and safe action"]
    A --> E["Evidence and limitations"]
    E --> R["Observed drill"]
    R --> V["Acceptance and revalidation"]
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Checkpoint:** The document-dump handoff fails before any repair is introduced: five of five records omit every capability-transfer field.

**Reflection bridge:** Rejecting the folder exposes the next failure. A richer package can still hide stale facts or quietly leave the FDE as the receiving owner, so the repair must be traceable to the frozen case.

---
## Part 1 - Build a Traceable Package

Load the frozen case instead of copying and silently editing its support facts. Then repair the package while preserving the difference between structural completeness and operational evidence.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Copy support facts into a private handoff sheet | Reference frozen IDs and retain source/version | Copies drift without a visible decision |
| Assign the FDE as receiving owner | Name the customer role with authority and escalation | Availability after handoff cannot depend on private context |
| Mark a drill `pass` because a runbook exists | Keep `not_run` until retained drill evidence exists | Authored procedure is not measured operator capability |

**Quick Health Check:** every package row needs artifact ID, capability, receiving owner, action, evidence reference, drill status, limitation, and revalidation trigger.

In [ ]:
# ── Load Frozen Facts and Repair Package Links ────────────────────────────────
from pathlib import Path
import json

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning" / "fde" / "shared").is_dir():
            return candidate
    raise FileNotFoundError("Run from inside the ai-portfolio checkout.")

repo_root = find_repo_root(Path.cwd().resolve())
fixture_path = repo_root / "learning/fde/shared/fixtures/riverside-engagement-v1.json"
engagement = json.loads(fixture_path.read_text(encoding="utf-8"))
support = engagement["support_and_handoff"]
open_unknowns = {item["unknown_id"]: item for item in engagement["unknowns"]}

repaired_package = [
    {
        "artifact_id": "OPS-DASH-01",
        "capability": "interpret health and choose an operating boundary",
        "receiving_owner": "ROLE-SUPPORT-L2",
        "decision_or_action": "continue, contain, stop ramp, or investigate",
        "evidence_ref": "dashboard and telemetry drill: not run",
        "drill_status": "not_run",
        "limitations": "live Azure signals and alert delivery are unvalidated",
        "revalidate_on": "signal, sampling, retention, release, or owner change",
    },
    {
        "artifact_id": "HOF-02",
        "capability": "contain, roll back, compensate, and re-enable",
        "receiving_owner": "ROLE-SUPPORT-L2",
        "decision_or_action": "execute the scenario runbook or escalate",
        "evidence_ref": "timed recovery drill: not run",
        "drill_status": "not_run",
        "limitations": "procedures are authored source, not rehearsed recovery evidence",
        "revalidate_on": "dependency, release, index, policy, runbook, or owner change",
    },
]
repaired_issues = assess_handoff(repaired_package)
print("PASS: required package fields are present." if not repaired_issues else repaired_issues)
print("BLOCKED: capability drills are still not run.")
print("  Takeaway: structural completeness and operational acceptance are separate gates.")

**Your turn:** Change one `receiving_owner` to `ROLE-FDE`. Predict that the exit rule will fail, then use the next cell to compare the actual owner scan with your prediction. Restore the customer owner afterward.

**Reflection bridge:** Structural fields now exist, but the package still has no live cloud evidence or completed drills. That residual failure forces a readiness decision over evidence, not document quality.

---
## Part 2 - Readiness Is a Decision Over Evidence

Riverside's platform source lacks live deployment evidence, committed SLOs, rollback rehearsal, approved residency, and reconciled production profiles. The readiness review must show those facts at the decision point.

```mermaid
flowchart LR
    A["Source and package records"] --> B["Critical readiness gates"]
    B --> C{"Scoped evidence accepted?"}
    C -- No --> D["BLOCKED with owner and next proof"]
    C -- Yes --> E["Bounded acceptance decision"]
    D --> F["Retain manual path and exposure limit"]
    E --> G["Revalidate on material change"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Treat source-only procedures as `[Measured]` recovery evidence | Mark them source-only until a scoped drill is retained | Provenance survives executive summaries |
| Average critical blockers into an overall readiness score | Keep identity, rollback, support, and ownership gates non-compensating | A polished majority cannot erase one unsafe boundary |
| Approve conditionally without expiry or exposure limit | Record owner, due date, limit, and automatic response | A condition becomes enforceable work |

**Quick Health Check**

- Every critical row has an accountable owner and decision authority.
- Every pass points to retained evidence with matching environment, release, region, and time scope.
- Every condition states exposure limits and automatic response.
- A failed drill blocks the decision even when the procedure looks correct.

In [ ]:
# ── Check Ownership and Apply an Evidence-Aware Readiness Gate ────────────────
# CHANGE THIS: set one receiving_owner to "ROLE-FDE" and predict the result.
fde_owned = [record["artifact_id"] for record in repaired_package
             if record["receiving_owner"] == "ROLE-FDE"]
readiness_rows = [
    {"gate": "live_azure_path", "state": "source_only", "critical": True},
    {"gate": "slo_capacity_cost", "state": "unset", "critical": True},
    {"gate": "rollback_rehearsal", "state": "not_run", "critical": True},
    {"gate": "support_after_hours", "state": open_unknowns["UNK-RIV-006"]["status"], "critical": True},
    {"gate": "post_hypercare_quality_owner", "state": open_unknowns["UNK-RIV-010"]["status"], "critical": True},
]
PASS_STATES = {"measured_pass", "customer_accepted"}
blockers = [row for row in readiness_rows if row["critical"] and row["state"] not in PASS_STATES]
print(f"FAIL: FDE remains receiving owner for {fde_owned}." if fde_owned
      else "PASS: no package capability is assigned to the FDE as receiving owner.")
print(f"BLOCKED: {len(blockers)} critical readiness gates lack accepted evidence.")
for blocker in blockers:
    print(f"  {blocker['gate']}: {blocker['state']}")
print("  Takeaway: source presence and planned procedures cannot pass an operational gate.")

---
## Part 3 - Dashboards and Alerts Must Produce Decisions

Generic uptime cannot tell you whether retrieval is current, an answer is grounded, a policy is denying correctly, a tool committed twice, or telemetry itself is blind.

```mermaid
flowchart LR
    S["Signal with scope and freshness"] --> T{"Threshold status known?"}
    T -- No --> B["Gather baseline; do not claim paging"]
    T -- Yes --> O["Named owner"]
    O --> A["First safe action"]
    A --> R["Runbook"]
    R --> E["Escalation and authority"]
    E --> C["Clear condition and evidence"]
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If `ALT-RIV-001` changes from `retain_all` to `random_1_percent`, will the alert remain safe because the event is rare, become cheaper but equivalent, or fail because a cross-tenant access signal may be sampled away? The next cell's safety rule resolves the named outcome.

**Your turn:** In a future authorized session, change only that sampling value, inspect the failure, then restore `retain_all`.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Dashboard CPU and call the service healthy | Pair service, quality, policy, tool, and telemetry-freshness signals | Infrastructure health cannot prove workflow safety |
| Page every signal without a first action | Bind owner, threshold, runbook, escalation, and clear condition | Alerts otherwise transfer ambiguity to on-call |
| Sample rare authorization failures | Retain all safety-critical events within approved privacy controls | Absence after sampling is not evidence of absence |

**Quick Health Check:** each signal has scope, freshness, decision, owner, blind spot, and evidence class; each alert adds first action, runbook, escalation, clear condition, and safety-appropriate retention.

**Reflection bridge:** Actionable alerts still fail if their runbook treats every failure as a restart. The PageTurn timeout now forces a precise distinction between stopping exposure and repairing committed state.

In [ ]:
# ── Link Signals and Alerts to Decisions ─────────────────────────────────────
dashboard_signals = [
    {"signal": "deadline_success", "decision": "stop ramp or isolate boundary", "owner": "ROLE-SUPPORT-L2", "blind_spot": "does not prove answer quality"},
    {"signal": "current_policy_retrieval", "decision": "freeze index or disable guidance", "owner": "ROLE-EDITORIAL-DIRECTOR", "blind_spot": "small slices may hide rare failures"},
    {"signal": "forbidden_access", "decision": "fail closed and engage Security", "owner": "ROLE-SECURITY", "blind_spot": "rare events must not be sampled away"},
    {"signal": "workflow_reconciliation", "decision": "pause writes and query committed state", "owner": "ROLE-SUPPORT-L2", "blind_spot": "timeout does not reveal commit state"},
    {"signal": "telemetry_freshness", "decision": "stop evidence-dependent changes", "owner": "ROLE-SUPPORT-L2", "blind_spot": "missing telemetry does not prove health"},
]
alerts = [
    {"alert_id": "ALT-RIV-001", "severity": "SEV-1", "owner": "ROLE-SECURITY", "first_action": "fail closed", "runbook": "RBK-RIV-IDENTITY", "sampling": "retain_all"},
    {"alert_id": "ALT-RIV-003", "severity": "SEV-2", "owner": "ROLE-SUPPORT-L2", "first_action": "pause writes and query state", "runbook": "RBK-RIV-TOOL-AMBIGUOUS", "sampling": "priority"},
]
signal_issues = [row["signal"] for row in dashboard_signals
                 if not all(row.get(field) for field in ("decision", "owner", "blind_spot"))]
alert_issues = []
for alert in alerts:
    missing = [field for field in ("owner", "first_action", "runbook") if not alert.get(field)]
    if alert["severity"] == "SEV-1" and alert["sampling"] != "retain_all":
        missing.append("retain_all safety evidence")
    if missing:
        alert_issues.append((alert["alert_id"], missing))
actual_sampling_outcome = "fail because a cross-tenant access signal may be sampled away" if alert_issues else "remain safe with retain_all"
print("PASS: signals map to decisions and alerts retain safety fields."
      if not signal_issues and not alert_issues else (signal_issues, alert_issues))
print(f"Prediction closed: alerts {actual_sampling_outcome}.")
print("  Takeaway: charts and notifications become controls only when they enable safe action.")

---
## Part 4 - Runbooks Separate Containment, Rollback, and Compensation

Riverside's PageTurn incident is the trap: the tool commits a workflow update, loses the response, and receives a retry. Rolling back model traffic does not undo the committed status change.

```mermaid
flowchart TD
    T["Trigger"] --> C["Contain and preserve evidence"]
    C --> K{"Outcome known?"}
    K -- No --> Q["Query target state"]
    Q --> A{"State now known?"}
    A -- No --> E["Escalate with evidence"]
    A -- Yes --> D{"Recovery type"}
    K -- Yes --> D
    D --> R["Rollback exposure"]
    D --> P["Compensate committed action"]
    D --> F["Approved degraded mode"]
    R --> V["Revalidate and obtain approval"]
    P --> V
    F --> V
    style T fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** A PageTurn write timed out after the remote system committed it. Does routing traffic to the previous model version restore the workflow, does a new retry key make the action safe, or must the team query the original business key and reconcile the committed state? The next cell maps the seeded tool incident to the bounded first action.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Retry every timeout | Query target state with the stable key | The far side may already have committed |
| Roll back traffic and declare recovery | Reconcile committed actions separately | Traffic changes cannot undo side effects |
| Use `compensation` for any cleanup | Reserve compensation for an explicit semantic counter-action; use correction or reconciliation when that is the real operation | Recovery language determines authority and audit evidence |
| Delete suspect deployments or logs | Preserve evidence and known-good state | Investigation and rollback need retained state |
| Re-enable after a health check | Run positive and negative gates with authority | Liveness is not safe behavior |

**Quick Health Check:** identify the failure boundary, stop new exposure, preserve the original business key, determine commit state, select rollback/correction/compensation precisely, retain the action history, and require re-enablement authority.

**Reflection bridge:** A correct runbook can still be invalidated by an unreviewed policy edit or an unfunded support promise. Recovery therefore feeds directly into controlled change and explicit service boundaries.

In [ ]:
# ── Map Seeded Incidents to Bounded First Actions ─────────────────────────────
runbook_map = {
    "policy": ("RBK-RIV-STALE-RETRIEVAL", "disable affected guidance and freeze index promotion"),
    "data": ("RBK-RIV-DATA-SYNC", "quarantine version and preserve lineage"),
    "identity": ("RBK-RIV-IDENTITY", "disable affected route and preserve access evidence"),
    "model": ("RBK-RIV-ROLLBACK", "require abstention and stop candidate exposure"),
    "tool": ("RBK-RIV-TOOL-AMBIGUOUS", "pause writes and query committed state"),
    "infrastructure": ("RBK-RIV-PROVIDER", "fail closed or use approved degraded mode"),
}
for incident in engagement["seeded_incidents"]:
    runbook_id, first_action = runbook_map[incident["domain"]]
    print(f"{incident['incident_id']} -> {runbook_id}: {first_action}")
tool_first_action = runbook_map["tool"][1]
print(f"Prediction closed: query and reconcile - '{tool_first_action}'.")
print("  Takeaway: the failure boundary selects the runbook; one restart guide cannot.")

---
## Parts 5 and 6 - Govern Change and Bound Support

Policy and threshold edits change authority or exposure. Treat them as versioned releases with positive and negative evaluation, approval, canary, rollback, and retained evidence. Support targets also need staffed hours, communication authority, vendor paths, and explicit exclusions.

```mermaid
flowchart LR
    R["Evidence-backed request"] --> I["Authority and impact"]
    I --> V["Versioned policy and tests"]
    V --> E["Positive and negative evaluation"]
    E --> A{"Authorized approval?"}
    A -- No --> H["Hold"]
    A -- Yes --> C["Bounded canary"]
    C --> G{"Observed gates pass?"}
    G -- No --> B["Rollback and preserve evidence"]
    G -- Yes --> P["Promote and monitor"]
    style R fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Riverside covers 08:00-18:00 Europe/London on weekdays, but `UNK-RIV-006` leaves out-of-hours coverage unresolved. `UNK-RIV-010` leaves post-hypercare model/retrieval quality ownership unresolved. Neither can be silently assigned to the FDE.

**Predict:** A stakeholder asks to lower a quality threshold because the canary is behind schedule. Will the change pass as an operational tweak, pass because rollback exists, or remain blocked until an authorized approver accepts versioned positive/negative evidence and bounded exposure? The next cell reports the failed gate by name.

**Your turn:** Write the evidence request, approval authority, exposure limit, automatic response, and retained decision that would make the proposal reviewable. Do not change the value until the gate exists.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Lower a gate to keep the calendar | Reopen the evidence and authority decision | Calendar pressure is not validation |
| Publish response targets without staffed coverage | Bind targets to hours, roles, vendors, and exclusions | An unfunded promise becomes incident debt |
| Leave post-hypercare quality with the FDE | Name a receiving owner and review cadence | Temporary delivery support becomes a hidden dependency |

**Quick Health Check:** policy changes carry artifact version, positive/negative cases, authority, canary, rollback target, retained decision, and revalidation trigger; support promises carry staffed hours, severities, communication authority, vendor paths, exclusions, and expiry.

**Reflection bridge:** Versioned controls and bounded support still prove nothing about operator behavior. The next gate is a drill in which a person must preserve safety and authority under pressure.

In [ ]:
# ── Gate a Policy Change and Expose Support Blockers ─────────────────────────
policy_change = {
    "change_id": "CHG-POL-RIV-001", "trigger": "INC-RIV-001",
    "versioned_artifact": True, "positive_cases": True, "negative_cases": True,
    "authorized_approval": False, "bounded_canary": True, "known_good_rollback": True,
}
required_change_gates = (
    "versioned_artifact", "positive_cases", "negative_cases",
    "authorized_approval", "bounded_canary", "known_good_rollback",
)
failed_change_gates = [gate for gate in required_change_gates if not policy_change[gate]]
support_checks = {
    "covered_hours_defined": bool(support["covered_hours"]),
    "severity_targets_defined": bool(support["severity_definitions"]),
    "after_hours_decided": open_unknowns["UNK-RIV-006"]["status"] != "open",
    "post_hypercare_quality_owner_decided": open_unknowns["UNK-RIV-010"]["status"] != "open",
}
support_blockers = [name for name, passed in support_checks.items() if not passed]
print(f"BLOCKED policy change: missing {failed_change_gates}." if failed_change_gates
      else "PASS: policy change may enter its approved cohort.")
print(f"BLOCKED support acceptance: missing {support_blockers}." if support_blockers
      else "PASS: support boundary and handback owner are decided.")
print("Prediction closed: remain blocked until authorized approval and bounded evidence exist.")
print("  Takeaway: source cannot approve itself, and the FDE cannot become unfunded permanent support.")

---
## Parts 7 and 8 - Drills Gate Acceptance and Exit

The minimum drills are trace correlation, rollback plus compensation, access revocation/deletion propagation, cross-tenant containment, and controlled threshold change. Attendance is not competence. A signature cannot override a failed critical drill, missing owner, unsupported claim, or unresolved support boundary.

```mermaid
flowchart LR
    A["Training attended"] --> B["Scenario drill"]
    B --> C{"Safety, authority, evidence pass?"}
    C -- No --> D["Reject acceptance and repair capability"]
    C -- Yes --> E["Retained drill record"]
    E --> F{"All critical gates pass?"}
    F -- No --> D
    F -- Yes --> G["Bounded sign-off and FDE exit"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Training agenda:** scope and limitations; architecture/version identity; dashboard and alert triage; evidence/privacy; containment and recovery; policy/threshold change; support and communications; recurring health and backlog.

**Scoring:** safety, authority, and evidence handling must be independently demonstrated. Do not average a critical failure away with fast timing on easier steps.

**Predict:** If the isolation drill is marked `pass` but `critical_failure=True` because the operator restored traffic without Security approval, does timing rescue the score, does a manager signature rescue it, or must acceptance remain blocked? The next cell exposes every blocker instead of averaging them.

**Your turn:** In a future authorized session, change only those two fields, compare the reported blocker with your prediction, then restore the authored `not_run` state.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Count attendance as competence | Score retained scenario performance | Presence does not prove safe action |
| Average a critical failure into a passing total | Keep safety, authority, and evidence non-compensating | Fast unsafe recovery is still unsafe |
| Sign off with FDE-only access intact | Remove private access and prove the customer path | Exit is incomplete while the service depends on delivery staff |

**Quick Health Check:** every drill records scenario, release/environment, operator, expected and observed actions, timing, critical errors, evidence handling, reviewer, remediation, and rerun result; acceptance remains blocked by any critical failure or unresolved owner.

**Reflection bridge:** Even a passing handoff decays as data, policy, models, thresholds, staffing, and value change. Ownership must therefore persist through recurring health decisions and an evidence-based backlog.

In [ ]:
# ── Keep Training, Acceptance, and FDE Exit Evidence-Gated ───────────────────
drills = [
    {"drill_id": "DRL-RIV-TRACE", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-ROLLBACK", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-DELETE", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-ISOLATION", "status": "not_run", "critical_failure": False},
    {"drill_id": "DRL-RIV-THRESHOLD", "status": "not_run", "critical_failure": False},
]
training_blockers = [drill["drill_id"] for drill in drills
                     if drill["status"] != "pass" or drill["critical_failure"]]
acceptance_inputs = {
    "readiness_blockers": len(blockers),
    "training_blockers": len(training_blockers),
    "support_blockers": len(support_blockers),
    "fde_owned_capabilities": len(fde_owned),
    "fde_only_access_removed": False,
    "authorized_signoff": False,
}
open_acceptance = {
    name: value for name, value in acceptance_inputs.items()
    if (isinstance(value, bool) and not value) or (isinstance(value, int) and value > 0)
}
print(f"BLOCKED training: {training_blockers}")
print(f"REJECT acceptance/FDE exit: {open_acceptance}" if open_acceptance
      else "PASS: bounded acceptance and FDE exit gates are satisfied.")
print("Prediction closed: critical failures remain non-compensating; timing and signature cannot rescue them.")
print("  Takeaway: elapsed time and signatures do not substitute for demonstrated ownership.")

---
## Part 9 - Recurring Health and the Evidence-Based Backlog

Handoff decays when workflow, data, policy, model, support, or ownership changes. Review customer value, retrieval/generation quality, identity/policy, reliability, capacity/cost, changes, training, and retirement together.

```mermaid
flowchart LR
    H["Health review"] --> E["Evidence, incidents, feedback, expiry"]
    E --> D{"Decision"}
    D --> C["Continue"]
    D --> X["Change or investigate"]
    D --> R["Reduce exposure or retire"]
    X --> B["Evidence-based backlog"]
    B --> G["Validation and rollout gate"]
    G --> H
    style H fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Review checks:** preserve claim classes; show expired evidence; reopen data, identity, evaluation, rollout, or handoff gates on material change; keep retirement available.

**Your turn:** Assume workflow value stays below the customer-approved floor while support burden exceeds the accepted envelope for three review periods. Write the `[Measured]` evidence, `[Modeled]` forward cost, authorized decision owner, and stop/revalidation conditions needed to choose among remediation, reduced exposure, deterministic replacement, and retirement.

**Common Pitfalls**

| Wrong | Right | Why it matters |
|---|---|---|
| Prioritize the loudest feature request | Rank by safety, authority, customer value, and evidence gap | Novelty can displace the actual operating risk |
| Carry expired evidence forward | Reopen the affected gate on material change | Acceptance has a scope and a shelf life |
| Treat retirement as failure | Compare ongoing value, risk, and support burden | Stopping can be the correct product decision |

**Quick Health Check:** every backlog item has trigger, affected criterion, claim class, owner, validation, exposure decision, stop condition, and next review; safety and authority blockers outrank feature demand.

**Reflection bridge:** The recurring review closes the operating loop. The notebook can now account for what was structurally built, what remains unmeasured, and why Riverside must still withhold acceptance.

In [ ]:
# ── Keep Backlog Priority Tied to Evidence ────────────────────────────────────
backlog = [
    {"id": "BLG-RIV-001", "trigger": "UNK-RIV-010", "safety_or_authority": True, "owner": "PER-RIV-001", "validation": "accepted post-hypercare ownership record"},
    {"id": "BLG-RIV-004", "trigger": "platform limitation: no rollback rehearsal", "safety_or_authority": True, "owner": "ROLE-IT-OWNER", "validation": "timed staging rollback and re-enablement evidence"},
    {"id": "BLG-RIV-006", "trigger": "INC-RIV-001", "safety_or_authority": False, "owner": "PER-RIV-002", "validation": "versioned retrieval and policy evaluation"},
]
backlog_issues = [item["id"] for item in backlog
                  if not all(item.get(field) for field in ("trigger", "owner", "validation"))]
ordered_backlog = sorted(backlog, key=lambda item: (not item["safety_or_authority"], item["id"]))
print("PASS: each item has trigger, owner, and validation." if not backlog_issues else backlog_issues)
print("Priority order:", [item["id"] for item in ordered_backlog])
print("  Takeaway: authority and safety blockers cannot be outvoted by feature demand.")

## Part 10 - Completed Roadmap, Coverage Ledger, and Honest Close

```mermaid
flowchart LR
    A["Document dump rejected"] --> B["Capability links authored"]
    B --> C["Readiness blockers exposed"]
    C --> D["Signals mapped to action"]
    D --> E["Recovery language separated"]
    E --> F["Change, support, and drills gated"]
    F --> G["Recurring ownership defined"]
    G --> H["Acceptance remains BLOCKED"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Completed roadmap

- **Failure exposed:** five of five document-dump records omit the fields needed to operate a capability.
- **Structural repair authored:** package rows now name owner, action, evidence reference, drill status, limitation, and revalidation trigger.
- **Riverside blockers preserved:** live Azure evidence, committed SLO/capacity/cost evidence, rollback rehearsal, out-of-hours support, and post-hypercare quality ownership remain unresolved.
- **Operational precision added:** dashboards map to decisions; alerts map to safe first actions; rollback stops exposure; reconciliation determines committed state; correction or compensation requires the corresponding business authority.
- **Acceptance decision:** `BLOCKED` in the authored scaffold because drills are `not_run`, critical readiness evidence is absent, private FDE access removal is unproven, and authorized sign-off is absent.

### Coverage ledger

| Tier | Coverage | Evidence boundary |
|---|---|---|
| Built and executed against synthetic fixtures | Capability validator, readiness gate, signal/alert map, incident-to-runbook map, policy/support gate, drill/exit gate, backlog ordering | The verified run reproduced `BLOCKED`, was cleared, and is not retained `[Measured]` operator or customer evidence |
| Explained and illustrated | Package traceability, dashboard blind spots, rollback versus reconciliation/correction/compensation, controlled change, support boundaries, drill scoring, recurring health and retirement | Mermaid flows and decision tables teach the operating method |
| Named with external validation required | Live telemetry delivery, threshold calibration, recovery timing, operator competence, customer acceptance, support staffing, vendor escalation, FDE access removal | Requires retained `[Measured]` drill or environment evidence and scoped `[Customer-validated]` decisions |

If a technique named in this notebook is absent from this ledger, treat that as a coverage defect.

### Key takeaways

1. Handoff transfers decisions and authority, not filenames.
2. Source completeness cannot be relabeled as measured operator capability.
3. A dashboard matters only when a scoped signal leads to a safe action and named owner.
4. Deployment rollback stops new exposure; it does not reverse a committed business action.
5. Use reconciliation, correction, and compensation precisely because each implies different evidence and authority.
6. Critical safety, authority, and evidence failures never average into acceptance.
7. FDE exit requires customer-owned access, drills, support, recurring review, and revalidation triggers.

> **Forward:** carry the unresolved Riverside blockers, capability evidence requirements, and `BLOCKED` acceptance decision into `09-capstone`, where the complete engagement package must make every dependency and claim traceable.